In [ ]:
import os
import pandas as pd
import numpy as np
import glob
pd.set_option('display.max_columns', None)

In [ ]:
#ADNI dataset
#"/project_cephfs/3022017.06/ADNI/phenotypes/Velocity files" --> may contain this. Also the surface holes = euler nr
root = "/project_cephfs/3022017.06/ADNI/freesurfer"

#aseg data
aseg_path = os.path.join(root, "aseg_stats.txt")
aseg_data = pd.read_csv(
    aseg_path,
    sep="\\s+",
    engine='python'
)

#lh data
lh_path = os.path.join(root, "lh_aparc_a2009s_stats.txt")
lh_data = pd.read_csv(
    lh_path,
    sep="\\s+",
    engine='python'
)

#rh data
rh_path = os.path.join(root, "rh_aparc_a2009s_stats.txt")
rh_data = pd.read_csv(
    rh_path,
    sep="\\s+",
    engine='python'
)

#merge left and right
lh_data = lh_data.rename(columns={"lh.aparc.a2009s.thickness" : "scan_id"})
rh_data = rh_data.rename(columns={"rh.aparc.a2009s.thickness" : "scan_id"})
lh_rh_data = pd.merge(lh_data, rh_data, on="scan_id", how="left")

#merge lh_rh and aseg
aseg_data = aseg_data.rename(columns={"Measure:volume" : "scan_id"})
adni_data = pd.merge(lh_rh_data, aseg_data, on="scan_id", how="left")
adni_data

#removing all the 'long' files
adni_data = adni_data[~adni_data['scan_id'].str.contains('long', case=False, na=False)]

#replacing the full name with the subj nr to be able to merge with the covariates dataset
pattern = r'\d{3}_S_\d{4}'

adni_data["id"] = adni_data["scan_id"]
adni_data["participant_id"] = adni_data["scan_id"].str.extract(f'({pattern})')
adni_data["Acq Date"] = adni_data["scan_id"].str.extract(r'(\d{8})')
adni_data["scan_id"] = adni_data["scan_id"].str.replace(r'^.*?(?=I\d+$)', '', regex=True)

adni_data["Acq Date"] = pd.to_datetime(adni_data["Acq Date"], format='%Y%m%d').dt.strftime('%m/%d/%Y')
adni_data["Acq Date dt"] = pd.to_datetime(adni_data["Acq Date"], format='%m/%d/%Y')
adni_data = adni_data.sort_values(by=["participant_id", "Acq Date dt"])


cols_to_move = ['scan_id', 'participant_id','Acq Date', 'Acq Date dt','id']
new_columns = cols_to_move + [col for col in adni_data.columns if col not in cols_to_move]
adni_data = adni_data[new_columns]

first_occurrences = adni_data.drop_duplicates(subset="participant_id", keep="first")

first_occurrences = first_occurrences.reset_index(drop=True)

#zoek baseline data

In [ ]:
#age/sex/site data
root = "/project_cephfs/3022017.06/ADNI/phenotypes"
asl_path = os.path.join(root, "Velocity_4_04_2025.csv")
asl_data = pd.read_csv(
    asl_path,
    sep=",",
    engine='python'
)

#age/sex/site data
root = "/project_cephfs/3022017.06/ADNI/phenotypes"
asl_path2 = os.path.join(root, "Velocity_MP-RAGE_4_04_2025.csv")
asl_data2 = pd.read_csv(
    asl_path2,
    sep=",",
    engine='python'
)

#removed Age here as an experiment to see whether the correct age still shows up later when we merge in the cells that creates the final dataset
asl_data = asl_data[['Image Data ID', 'Subject','Group', 'Sex', 'Acq Date']]
asl_data2 = asl_data2[['Image Data ID', 'Subject','Group', 'Sex', 'Acq Date']]

cov_data = pd.concat([asl_data, asl_data2], axis=0)
cov_data = cov_data.rename(columns={"Image Data ID" : "scan_id"})
cov_data = cov_data.rename(columns={"Subject" : "participant_id"})
cov_data['Acq Date'] = pd.to_datetime(cov_data['Acq Date']).dt.strftime('%m/%d/%Y')



merged_data = cov_data.merge(first_occurrences, on=["participant_id","Acq Date","scan_id"], how="right")
# merged_data = merged_data.rename(columns={'participant_id_x':'participant_id'})

# merged_data
#cov_data = cov_data[cov_data['participant_id'] == "002_S_0295"]
merged_data = merged_data.drop(columns=['Acq Date', 'id'])
merged_data = merged_data.rename(columns={"Acq Date dt": "Acq Date"})
merged_data

In [ ]:
adni_merge_df = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/ADNIMERGE_02May2025.csv')
adni_merge_df = adni_merge_df.rename(columns={"PTID":"participant_id", "EXAMDATE":"Acq Date","SITE":"Site","AGE":"Age", "PTGENDER":"Sex"})
adni_merge_df = adni_merge_df.replace({'Female': 'F', 'Male': 'M'})
adni_merge_df = adni_merge_df[["participant_id","Site","Age"]]


asl_adni_merge = adni_merge_df.merge(merged_data, on=['participant_id'], how='right')
asl_adni_merge = asl_adni_merge.drop_duplicates(subset="participant_id", keep="first")
asl_adni_merge = asl_adni_merge.reset_index(drop=True)
asl_adni_merge = asl_adni_merge.dropna()
asl_adni_merge = asl_adni_merge.drop(columns=['scan_id','Acq Date'])
asl_adni_merge = asl_adni_merge.rename(columns={"Site_x":"Site"})
asl_adni_merge

In [ ]:
asl_adni_merge.to_csv("ADNI_data.csv")

In [ ]:
ADNI_dxsum = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/All_Subjects_DXSUM_02May2025.csv')
# ADNI_dxsum = ADNI_dxsum[["PTID", "VISCODE", "VISCODE2", "DIAGNOSIS", "EXAMDATE"]]
ADNI_dxsum = ADNI_dxsum.rename(columns={'PTID': 'Subject', 'EXAMDATE':'Acq Date'})

ADNI_dxsum['Acq Date'] = pd.to_datetime(ADNI_dxsum['Acq Date']).dt.strftime('%m/%d/%Y')

ADNI_dxsum


In [ ]:
ADNI_demo1 = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/Velocity_4_04_2025.csv')
ADNI_demo2 = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/Velocity_MP-RAGE_4_04_2025.csv')
ADNI_demo = pd.concat([ADNI_demo1, ADNI_demo2], axis = 0)
ADNI_demo.rename(columns={"Image Data ID": "IID", "Visit":"VISCODE"}, inplace ="True")
ADNI_demo

In [ ]:
MRI_info_1 = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/MRI3META_19May2025.csv')
MRI_info_1= MRI_info_1[["PTID", "VISCODE", "VISCODE2"]]
MRI_info_2 = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/MRIMETA_19May2025.csv')
MRI_info_2= MRI_info_2[["PTID", "VISCODE", "VISCODE2"]]

MRI_info = pd.concat([MRI_info_1, MRI_info_2], axis =0)
MRI_info.rename(columns={"PTID": "Subject"}, inplace=True)

ADNI_demo = ADNI_demo.merge(MRI_info, on=['Subject', 'VISCODE'], how ="left")
ADNI_demo_full = ADNI_demo.merge(ADNI_dxsum[["Subject", "VISCODE2","DIAGNOSIS"]],
                                 on=['Subject', 'VISCODE2'],
                                 how='left')




In [ ]:
ADNI_demo_full = ADNI_demo_full.dropna()
ADNI_demo_full

In [ ]:

#ADNI
ADNI_aseg = pd.read_csv("/project_cephfs/3022017.06/ADNI/freesurfer/aseg_stats.txt", sep = "\t")
ADNI_lh_DES = pd.read_csv("/project_cephfs/3022017.06/ADNI/freesurfer/lh_aparc_a2009s_stats.txt", sep = "\t")
ADNI_rh_DES = pd.read_csv("/project_cephfs/3022017.06/ADNI/freesurfer/rh_aparc_a2009s_stats.txt", sep = "\t")

ADNI_lh_DK = pd.read_csv("/project_cephfs/3022017.06/ADNI/freesurfer/lh_aparc_stats.txt", sep = "\t")
ADNI_rh_DK = pd.read_csv("/project_cephfs/3022017.06/ADNI/freesurfer/rh_aparc_stats.txt", sep = "\t")

ADNI_lh_DK = ADNI_lh_DK.rename(columns={"lh.aparc.thickness": "ID"})
ADNI_rh_DK = ADNI_rh_DK.rename(columns={"rh.aparc.thickness": "ID"})

ADNI_DK = ADNI_lh_DK.merge(ADNI_rh_DK, on="ID")

ADNI_lh_DES = ADNI_lh_DES.rename(columns={"lh.aparc.a2009s.thickness": "ID"})
ADNI_rh_DES = ADNI_rh_DES.rename(columns={"rh.aparc.a2009s.thickness": "ID"})

ADNI_DES = ADNI_lh_DES.merge(ADNI_rh_DES, on="ID")

ADNI_DES = ADNI_DES.loc[:, ~ADNI_DES.columns.str.contains("eTIV")]
ADNI_DES = ADNI_DES.loc[:, ~ADNI_DES.columns.str.contains("BrainSegVolNotVent")]

ADNI_DK = ADNI_DK.loc[:, ~ADNI_DK.columns.str.contains("eTIV")]
ADNI_DK = ADNI_DK.loc[:, ~ADNI_DK.columns.str.contains("BrainSegVolNotVent")]

ADNI_aseg = ADNI_aseg.rename(columns={"Measure:volume": "ID"})
# we need all variables that were NOT processed using longitudian freesurfer
ADNI_aseg_long = ADNI_aseg[~ADNI_aseg["ID"].str.contains(r"\.long\.", regex=True)].copy()
ADNI_DES_long = ADNI_DES[~ADNI_DES["ID"].str.contains(r"\.long\.", regex=True)].copy()
ADNI_DK_long = ADNI_DK[~ADNI_DK["ID"].str.contains(r"\.long\.", regex=True)]


pattern = r'_I[^_]*'

ADNI_aseg_long["IID"] = ADNI_aseg_long["ID"].str.extract(r'_((I[^.]*))').iloc[:, 0]
ADNI_DES_long["IID"] = ADNI_DES_long["ID"].str.extract(r'_((I[^.]*))').iloc[:, 0]
ADNI_DK_long["IID"] = ADNI_DK_long["ID"].str.extract(r'_((I[^.]*))').iloc[:, 0]

ADNI_DES_long = ADNI_DES_long[~ADNI_DES_long['ID'].str.contains("real", na=False)]
ADNI_DK_long = ADNI_DK_long[~ADNI_DK_long['ID'].str.contains("real", na=False)]
ADNI_aseg_long = ADNI_aseg_long[~ADNI_aseg_long["IID"].str.contains("real", na=False)]

ADNI_aseg_long["IID"] = ADNI_aseg_long["IID"].astype(str)
#%%
ADNIMERGE = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/ADNIMERGE_02May2025.csv')
ADNIMERGE = ADNIMERGE.rename(columns={'PTID': 'Subject', 'VISCODE': 'VISCODE2'})

ADNI_conversion_info = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/All_Subjects_DXSUM_02May2025.csv')
ADNI_conversion_info = ADNI_conversion_info[["PTID", "VISCODE", "VISCODE2", "DIAGNOSIS"]]
ADNI_conversion_info = ADNI_conversion_info.rename(columns={'PTID': 'Subject'})

ADNI_demo1 = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/Velocity_4_04_2025.csv')
ADNI_demo2 = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/Velocity_MP-RAGE_4_04_2025.csv')
ADNI_demo = pd.concat([ADNI_demo1, ADNI_demo2], axis = 0)
ADNI_demo.rename(columns={"Image Data ID": "IID", "Visit":"VISCODE"}, inplace ="True")
ADNI_demo["Sex"] = ADNI_demo["Sex"].replace({"M": 1, "F": 0})


MRI_info_1 = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/MRI3META_19May2025.csv')
MRI_info_1= MRI_info_1[["PTID", "VISCODE", "VISCODE2"]]
MRI_info_2 = pd.read_csv('/project_cephfs/3022017.06/ADNI/phenotypes/MRIMETA_19May2025.csv')
MRI_info_2= MRI_info_2[["PTID", "VISCODE", "VISCODE2"]]

MRI_info = pd.concat([MRI_info_1, MRI_info_2], axis =0)
MRI_info.rename(columns={"PTID": "Subject"}, inplace=True)

ADNI_demo = ADNI_demo.merge(MRI_info, on=['Subject', 'VISCODE'], how ="left")
#%%
ADNI_demo_full = ADNI_demo.merge( ADNI_conversion_info[["Subject", "VISCODE2","DIAGNOSIS"]],
                                 on=['Subject', 'VISCODE2'],
                                 how='left')
#remove werid ID
ADNI_demo_full["VISCODE"].value_counts()